# Autoencoder

In [44]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

In [45]:
def to_conv(x):
    # Conv1d needs channels (the features) in the middle
    return np.transpose(x, (0,2,1)).astype(np.float32)

## Load Data

In [46]:
# check whether to use cpu or gpu for training
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# load data
d = np.load(os.path.join("data","dataset.npz"), allow_pickle=True)
FEATURES = list(d["features"])
GRID = d["grid"]
X_train, id_train = to_conv(d["X_train"]), d["id_train"]
X_test, id_test = to_conv(d["X_test"]), d["id_test"]

# training settings
BATCH_SIZE = 256
ALPHA = 0.34 # LeakyReLU negative slope
EPOCHS = 1
LR = 1e-3
LATENT_DIM = 4

In [47]:
# split training set in training and validation
rng = np.random.default_rng(7)
train_stars = np.unique(id_train)
n_val = max(1, round(0.15 * len(train_stars)))
val_stars = rng.choice(train_stars, size=n_val, replace=False)
is_val = np.isin(id_train, val_stars)
X_tr, X_val = X_train[~is_val], X_train[is_val]

In [48]:
# create the data loader for the training, validation and test sets
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_tr)),
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val)),
    batch_size=2*BATCH_SIZE
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test)),
    batch_size=2*BATCH_SIZE
)

## Model

In [49]:
class ConvAutoencoder(nn.Module):
    def __init__(self, n_features=4, n_points=100, latent_dim=4):
        super().__init__()

        self.encoder = Encoder(n_features, n_points, latent_dim)
        sizes = list( reversed(self.encoder.lengths[:-1]) ) + [n_points]
        self.decoder = Decoder(n_features, latent_dim, self.encoder.conv_len, sizes)
    
    def forward(self, x):
        return self.decoder(self.encoder(x))


class Encoder(nn.Module):
    def __init__(self, n_features, n_points, latent_dim):
        super().__init__()

        # convolutional layers (100->50->25->13)
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, 16, kernel_size=5, stride=2, padding=2), 
            nn.LeakyReLU(ALPHA),
            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2), 
            nn.LeakyReLU(ALPHA),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2), 
            nn.LeakyReLU(ALPHA),
        )

        # fake forward to save the lengths after conv layers
        with torch.no_grad():
            x = torch.zeros(1, n_features, n_points)
            self.lengths = []
            for layer in self.conv:
                x = layer(x)
                if isinstance(layer, nn.Conv1d):
                    self.lengths.append( x.shape[-1] )

        # save convolutional length for decoder
        self.conv_len = self.lengths[-1]

        # fully connect output of conv layers with latent space
        self.fc = nn.Linear(64*self.conv_len, latent_dim)
    
    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))


class Decoder(nn.Module):
    def __init__(self, n_features, latent_dim, conv_len, sizes):
        super().__init__()
        self.conv_len = conv_len

        # fully connect latent dim to flattened deconvoluted vector
        self.fc = nn.Linear(latent_dim, 64*conv_len)
        # smooth upsample + convolution
        self.up = nn.Sequential(
            nn.Upsample(size=sizes[0], mode="linear", align_corners=False),
            nn.ConvTranspose1d(64, 32, 5, padding=2),
            nn.LeakyReLU(ALPHA),
            nn.Upsample(size=sizes[1], mode="linear", align_corners=False),
            nn.ConvTranspose1d(32, 16, 5, padding=2),
            nn.LeakyReLU(ALPHA),
            nn.Upsample(size=sizes[2], mode="linear", align_corners=False),
            nn.ConvTranspose1d(16, n_features, 5, padding=2),
            nn.Sigmoid(),
        )
    
    def forward(self, z):
        return self.up(self.fc(z).view(-1, 64, self.conv_len))

## Training

In [50]:
def train_ae(model, train_loader, val_loader):
    model.to(DEVICE)

    # optimizer
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    # loss function
    loss_fn = nn.MSELoss()

    # loss history
    history = {"train": [], "val": []}
    best_val, best_state = float("inf"), None

    for epoch in range(EPOCHS):
        # training step
        model.train()
        
        loss_train = 0.0
        for (xb,) in train_loader:
            xb = xb.to(DEVICE)
            opt.zero_grad()

            # calculate loss
            loss = loss_fn(model(xb), xb)

            # backpropagation and update weights
            loss.backward()
            opt.step()

            # save loss
            loss_train += loss.item() * xb.size(0)
        loss_train /= len(train_loader.dataset)

        # validation step
        model.eval()
        loss_val = 0.0
        with torch.no_grad():
            for (xb,) in val_loader:
                xb = xb.to(DEVICE)
                
                # calculate loss
                loss = loss_fn(model(xb), xb)

                # save loss
                loss_val += loss.item() * xb.size(0)
        loss_val /= len(val_loader.dataset)

        # save current epoch training and validation loss
        history["train"].append(loss_train)
        history["val"].append(loss_val)

        # keep the model with best loss_val
        if loss_val < best_val:
            best_val = loss_val
            best_state = {
                k: v.cpu().clone() for k,v in model.state_dict().items()
            }
        if (epoch+1) % 5 == 0:
            print(f"Epoch: {epoch+1:3d} | Train: {loss_train:.5f} | Val: {loss_val:.5f}")
        
    model.load_state_dict(best_state)
    
    return history

In [ ]:
model = ConvAutoencoder(
    n_features=len(FEATURES),
    n_points=len(GRID),
    latent_dim=LATENT_DIM
)

history = train_ae(model, train_loader, val_loader)